In [1]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np

sales = pd.read_pickle("../data/processed/nyc_sales_2003_2025_matched.pkl")
print(sales.shape)

bands = [-1, 0, 1_000, 10_000, 50_000, 100_000, 250_000, 500_000,
         1_000_000, 5_000_000, 50_000_000, np.inf]
print(pd.cut(sales['SALE PRICE'], bands).value_counts().sort_index())

low = sales.loc[sales['SALE PRICE'].between(1, 100_000), 'SALE PRICE']
print("\nMost common prices between $1 and $100K:")
print(low.value_counts().head(15))

print("\nShare of $0 sales by year:")
print((sales['SALE PRICE'].eq(0).groupby(sales['SALE DATE'].dt.year).mean() * 100).round(1).to_string())

(2070484, 36)
SALE PRICE
(-1.0, 0.0]                622730
(0.0, 1000.0]               29488
(1000.0, 10000.0]            7661
(10000.0, 50000.0]          33849
(50000.0, 100000.0]         32048
(100000.0, 250000.0]       169651
(250000.0, 500000.0]       399141
(500000.0, 1000000.0]      467364
(1000000.0, 5000000.0]     262692
(5000000.0, 50000000.0]     42298
(50000000.0, inf]            3562
Name: count, dtype: int64

Most common prices between $1 and $100K:
SALE PRICE
10.0        17300
1.0          4185
100000.0     3655
100.0        2789
1000.0       2695
50000.0      1816
75000.0      1644
10000.0      1555
80000.0      1540
90000.0      1492
20000.0      1322
25000.0      1284
85000.0      1276
95000.0      1229
60000.0      1205
Name: count, dtype: int64

Share of $0 sales by year:
SALE DATE
2003    30.9
2004    26.9
2005    25.7
2006    27.0
2007    27.8
2008    30.9
2009    32.9
2010    32.4
2011    31.3
2012    34.6
2013    30.3
2014    29.6
2015    29.0
2016    30.3
2017  

In [2]:
sales['CAT_CODE'] = sales['BUILDING CLASS CATEGORY'].str.strip().str[:2]
sales['IS_ZERO'] = sales['SALE PRICE'].eq(0)

cat = sales.groupby('CAT_CODE').agg(
    example=('BUILDING CLASS CATEGORY', 'first'),
    n=('SALE PRICE', 'size'),
    median_price=('SALE PRICE', 'median'),
    pct_zero=('IS_ZERO', 'mean'),
)
cat['pct_zero'] = (cat['pct_zero'] * 100).round(1)
print(cat.sort_values('n', ascending=False).to_string())

mid = sales[sales['SALE PRICE'].between(10_001, 50_000)]
print("\nWhat sells for $10K-$50K:")
print(mid['CAT_CODE'].value_counts().head(10))

                                               example       n  median_price  pct_zero
CAT_CODE                                                                              
01        01  ONE FAMILY HOMES                          443349      343000.0      33.3
02        02  TWO FAMILY HOMES                          395222      335000.0      39.1
10        10  COOPS - ELEVATOR APARTMENTS               310402      340000.0       9.5
13        13  CONDOS - ELEVATOR APARTMENTS              293852      661862.0      21.4
03        03  THREE FAMILY HOMES                        111737      235000.0      44.6
07        07  RENTALS - WALKUP APARTMENTS                86841      260000.0      44.6
09        09  COOPS - WALKUP APARTMENTS                  61105      263000.0       7.7
28        28  COMMERCIAL CONDOS                          39159       10035.0      41.6
04        04  TAX CLASS 1 CONDOS                         38025      315000.0      25.6
05        05  TAX CLASS 1 VACANT LAND      

In [3]:
SEGMENTS = {**{c: 'HOUSE' for c in ['01', '02', '03']},
            **{c: 'CONDO_COOP' for c in ['04', '09', '10', '12', '13', '15', '17']},
            **{c: 'MULTIFAMILY' for c in ['07', '08', '14']}}
sales['SEGMENT'] = sales['CAT_CODE'].map(SEGMENTS)

# Funnel: how many sales survive each step
s1 = sales[sales['SEGMENT'].notna()]
s2 = s1[s1['SALE PRICE'] >= 10_000]
size = s2.groupby('MATCH_GROUP')['MATCH_GROUP'].transform('size')
s3 = s2[size >= 2].sort_values(['MATCH_GROUP', 'SALE DATE', 'SALE PRICE'])
print(f"All sales                       : {len(sales):,}")
print(f"Residential segments            : {len(s1):,}")
print(f"...price >= $10,000             : {len(s2):,}")
print(f"...property has 2+ such sales   : {len(s3):,}")

# Consecutive pairs within each property
g = s3.groupby('MATCH_GROUP')
pairs = s3[['MATCH_GROUP', 'SEGMENT', 'BOROUGH', 'NEIGHBORHOOD', 'BBL', 'SALE DATE', 'SALE PRICE']].copy()
pairs['date0']  = g['SALE DATE'].shift(1)
pairs['price0'] = g['SALE PRICE'].shift(1)
pairs = pairs.dropna(subset=['date0']).rename(columns={'SALE DATE': 'date1', 'SALE PRICE': 'price1'})
pairs['days'] = (pairs['date1'] - pairs['date0']).dt.days
pairs['log_ratio'] = np.log(pairs['price1'] / pairs['price0'])

print(f"\nRaw consecutive pairs: {len(pairs):,}")
print(pairs['SEGMENT'].value_counts())
bins = [-1, 0, 30, 90, 180, 365, 365 * 2, 365 * 5, 365 * 25]
print("\nDays between sales:")
print(pd.cut(pairs['days'], bins).value_counts().sort_index())
print("\nLog price ratio quantiles:")
print(pairs['log_ratio'].quantile([.001, .01, .05, .25, .5, .75, .95, .99, .999]).round(2))

All sales                       : 2,070,484
Residential segments            : 1,847,060
...price >= $10,000             : 1,298,117
...property has 2+ such sales   : 766,875

Raw consecutive pairs: 540,717
SEGMENT
CONDO_COOP     332579
HOUSE          188650
MULTIFAMILY     19488
Name: count, dtype: int64

Days between sales:
days
(-1, 0]          32178
(0, 30]         107454
(30, 90]         46460
(90, 180]        36404
(180, 365]       42959
(365, 730]       44928
(730, 1825]      80465
(1825, 9125]    149869
Name: count, dtype: int64

Log price ratio quantiles:
0.001   -4.05
0.010   -1.78
0.050   -0.79
0.250   -0.06
0.500    0.17
0.750    0.46
0.950    1.07
0.990    2.19
0.999    4.10
Name: log_ratio, dtype: float64


In [4]:
print(pairs['log_ratio'].quantile([.001, .01, .05, .25, .5, .75, .95, .99, .999]).round(2).to_string())

pairs['days_bucket'] = pd.cut(pairs['days'], bins)
print("\nLog-ratio behavior by interval:")
print(pairs.groupby('days_bucket', observed=True)['log_ratio'].agg(
    median='median', std='std', share_abs_gt_1=lambda s: (s.abs() > 1).mean()).round(3))

short = pairs[pairs['days'] <= 30].join(sales[['ADDRESS', 'ADDRESS_UNIT', 'APARTMENT NUMBER']])
mask = short['ADDRESS_UNIT'].isna() & short['APARTMENT NUMBER'].notna()
print(f"\nShort pairs (<=30 days): {len(short):,}")
print(f"...unit missing from address but APARTMENT NUMBER present: {mask.sum():,} ({mask.mean():.1%})")
print(short.loc[mask, ['ADDRESS', 'APARTMENT NUMBER', 'date0', 'date1', 'price0', 'price1']]
      .sample(8, random_state=1).to_string())

0.001   -4.05
0.010   -1.78
0.050   -0.79
0.250   -0.06
0.500    0.17
0.750    0.46
0.950    1.07
0.990    2.19
0.999    4.10

Log-ratio behavior by interval:
              median    std  share_abs_gt_1
days_bucket                                
(-1, 0]        0.143  0.497           0.059
(0, 30]       -0.022  0.622           0.082
(30, 90]       0.018  0.661           0.089
(90, 180]      0.164  0.660           0.084
(180, 365]     0.270  0.674           0.098
(365, 730]     0.214  0.711           0.103
(730, 1825]    0.185  0.666           0.086
(1825, 9125]   0.304  0.656           0.099

Short pairs (<=30 days): 139,632
...unit missing from address but APARTMENT NUMBER present: 126,866 (90.9%)
                                           ADDRESS APARTMENT NUMBER      date0      date1     price0     price1
316147   2140 OCEAN AVENUE                             1A           2010-10-18 2010-10-26   286455.0   466530.0
1378690                            55 AUSTIN PLACE               2S 

In [5]:
import importlib, src.matching as m
importlib.reload(m)

sales['UNIT_FINAL'] = [m.clean_unit(a, b) for a, b in zip(sales['ADDRESS_UNIT'], sales['APARTMENT NUMBER'])]

had = sales['ADDRESS_UNIT'].notna()
now = sales['UNIT_FINAL'].notna()
print(f"Unit known before: {had.sum():,} | after: {now.sum():,} | newly filled: {(now & ~had).sum():,}")
both = had & sales['APARTMENT NUMBER'].notna()
agree = (sales.loc[both, 'ADDRESS_UNIT'] == sales.loc[both, 'APARTMENT NUMBER'].map(m.normalize_unit)).mean()
print(f"Where both exist, they agree: {agree:.1%}")

sales['MATCH_GROUP_V2'], stats = m.build_match_groups(sales, 'UNIT_FINAL')
print(stats)
print("Distinct groups  old:", f"{sales['MATCH_GROUP'].nunique():,}", " new:", f"{sales['MATCH_GROUP_V2'].nunique():,}")
print(m.evaluate_matching(sales, 'UNIT_FINAL', group_col='MATCH_GROUP_V2'))

Unit known before: 464,179 | after: 837,930 | newly filled: 374,559
Where both exist, they agree: 30.2%
{'candidates': 2037786, 'rescued': 427, 'ambiguous': 118}
Distinct groups  old: 1,060,139  new: 1,218,550
{'rows': 814658, 'P': 540562, 'TP': 538094, 'T': 543126, 'precision': 0.9954, 'recall_fair': 0.9907}


In [6]:
both = sales['ADDRESS_UNIT'].notna() & sales['APARTMENT NUMBER'].notna()
d = sales.loc[both, ['ADDRESS', 'ADDRESS_UNIT', 'APARTMENT NUMBER', 'SALE DATE']].copy()
d['APT_NORM'] = d['APARTMENT NUMBER'].map(m.normalize_unit)

import re
loose = lambda s: re.sub(r'[^A-Z0-9]', '', str(s).upper()).lstrip('0')
d['strict'] = d['ADDRESS_UNIT'] == d['APT_NORM']
d['loose']  = d['ADDRESS_UNIT'].map(loose) == d['APT_NORM'].map(loose)

print(f"rows with both: {len(d):,}")
print(f"strict agree: {d['strict'].mean():.1%} | loose agree (ignore punctuation/zeros): {d['loose'].mean():.1%}")

print("\nAgreement by sale year:")
print(d.groupby(d['SALE DATE'].dt.year)['loose'].agg(['mean', 'size']).round(3).to_string())

print("\nSample of loose disagreements:")
print(d[~d['loose']].sample(25, random_state=1)[['ADDRESS', 'ADDRESS_UNIT', 'APARTMENT NUMBER']].to_string())

rows with both: 323,278
strict agree: 30.2% | loose agree (ignore punctuation/zeros): 30.3%

Agreement by sale year:
            mean   size
SALE DATE              
2003       0.000  18857
2004       0.000  22458
2005       0.000  22950
2006       0.000  20281
2007       0.000  20262
2008       0.000  16959
2009       0.000  13188
2010       0.000   5909
2011       0.000  13695
2012       0.000  15359
2013       0.000  18452
2014       0.000  17858
2015       0.000  18111
2016       0.000      2
2017       0.333      6
2018       0.987    686
2019       0.990  17641
2020       0.000      1
2022       0.989  23440
2023       0.990  17877
2024       0.988  18657
2025       0.994  20629

Sample of loose disagreements:
                                           ADDRESS ADDRESS_UNIT APARTMENT NUMBER
1011052  87-50 204TH   STREET, B34                          B34                 
653952   5 TUDOR CITY PLACE, 1732                          1732                 
447354   573 GRAN ST, APT D704  

In [7]:
print(repr(m.normalize_unit('      ')), repr(m.normalize_unit('')), repr(m.normalize_unit(None)))

uf = sales['UNIT_FINAL']
print("blank-ish UNIT_FINAL:", (uf.notna() & (uf.astype(str).str.strip() == '')).sum())

# True agreement among rows where APARTMENT NUMBER is a real value
real_apt = sales['APARTMENT NUMBER'].astype(str).str.strip().replace('', pd.NA)
both2 = sales['ADDRESS_UNIT'].notna() & real_apt.notna()
d2 = sales.loc[both2]
print(f"rows with both (real): {both2.sum():,}")
print(f"agree: {(d2['ADDRESS_UNIT'] == real_apt[both2].map(m.normalize_unit)).mean():.1%}")

# Coverage after the fix, condo/co-op sales only, by year
co = sales[sales['SEGMENT'] == 'CONDO_COOP'] if 'SEGMENT' in sales else sales[sales['BUILDING CLASS CATEGORY'].str.strip().str[:2].isin(['04','09','10','12','13','15','17'])]
print((co.groupby(co['SALE DATE'].dt.year)['UNIT_FINAL'].apply(lambda s: s.notna().mean())).round(3).to_string())

None None None
blank-ish UNIT_FINAL: 0
rows with both (real): 99,622
agree: 98.1%
SALE DATE
2003    0.928
2004    0.932
2005    0.971
2006    0.979
2007    0.978
2008    0.980
2009    0.981
2010    0.979
2011    0.983
2012    0.983
2013    0.981
2014    0.975
2015    0.977
2016    0.947
2017    0.934
2018    0.981
2019    0.991
2020    0.989
2021    0.521
2022    0.992
2023    0.991
2024    0.991
2025    0.992


In [8]:
sales['MATCH_GROUP_OLD'] = sales['MATCH_GROUP']
sales['MATCH_GROUP'] = sales['MATCH_GROUP_V2']
sales.to_pickle("../data/processed/nyc_sales_2003_2025_matched_v2.pkl")   # old pickle stays as backup

CC = ['04','09','10','12','13','15','17']
sales['CAT_CODE'] = sales['BUILDING CLASS CATEGORY'].str.strip().str[:2]
co21 = sales[(sales['CAT_CODE'].isin(CC)) & (sales['SALE DATE'].dt.year == 2021)]
print("2021 condo/co-op rows:", len(co21), "| no unit:", co21['UNIT_FINAL'].isna().sum())
print(co21.groupby('BOROUGH')['UNIT_FINAL'].apply(lambda s: s.isna().mean()).round(2).to_string())
print(co21.groupby(co21['SALE DATE'].dt.month)['UNIT_FINAL'].apply(lambda s: s.isna().mean()).round(2).to_string())
print(co21[co21['UNIT_FINAL'].isna()][['ADDRESS', 'APARTMENT NUMBER', 'SALE DATE', 'SALE PRICE']].sample(10, random_state=1).to_string())

2021 condo/co-op rows: 41766 | no unit: 19999
BOROUGH
1    0.49
2    0.67
3    0.33
4    0.59
5    0.44
SALE DATE
1     0.51
2     0.51
3     0.50
4     0.47
5     0.47
6     0.46
7     0.48
8     0.47
9     0.49
10    0.50
11    0.49
12    0.42
                             ADDRESS APARTMENT NUMBER  SALE DATE  SALE PRICE
1416114              62 HICKS STREET              NaN 2021-01-22    840000.0
1399342         303 WEST 66TH STREET              NaN 2021-04-29    975000.0
1451220             110-21 73RD ROAD              NaN 2021-11-22    342000.0
1395690             1120 PARK AVENUE              NaN 2021-06-03   1895000.0
1384220           60 EAST 8TH STREET              NaN 2021-08-23   1180000.0
1433197               198 6TH AVENUE              NaN 2021-07-19   2476000.0
1447522               138-18 28TH RD              NaN 2021-08-02    207000.0
1399542          150 WEST END AVENUE              NaN 2021-05-20    755000.0
1451540  27010 GRAND CENTRAL PARKWAY              NaN 2021-08

In [9]:
SEGMENTS = {**{c:'HOUSE' for c in ['01','02','03']},
            **{c:'CONDO_COOP' for c in CC},
            **{c:'MULTIFAMILY' for c in ['07','08','14']}}
sales['SEGMENT'] = sales['CAT_CODE'].map(SEGMENTS)

s1 = sales[sales['SEGMENT'].notna()]
no_unit_cc = (s1['SEGMENT'] == 'CONDO_COOP') & s1['UNIT_FINAL'].isna()
print(f"Segment rows: {len(s1):,} | condo/co-op without a unit, dropped: {no_unit_cc.sum():,}")
s1 = s1[~no_unit_cc]
s2 = s1[s1['SALE PRICE'] >= 10_000]
size = s2.groupby('MATCH_GROUP')['MATCH_GROUP'].transform('size')
s3 = s2[size >= 2].sort_values(['MATCH_GROUP', 'SALE DATE', 'SALE PRICE'])

g = s3.groupby('MATCH_GROUP')
pairs = s3[['MATCH_GROUP','SEGMENT','BOROUGH','NEIGHBORHOOD','BBL','SALE DATE','SALE PRICE']].copy()
pairs['date0'] = g['SALE DATE'].shift(1); pairs['price0'] = g['SALE PRICE'].shift(1)
pairs = pairs.dropna(subset=['date0']).rename(columns={'SALE DATE':'date1','SALE PRICE':'price1'})
pairs['days'] = (pairs['date1'] - pairs['date0']).dt.days
pairs['log_ratio'] = np.log(pairs['price1'] / pairs['price0'])
print("pairs:", f"{len(pairs):,}", "(before fix: see Step 28)")

bins = [-1,0,30,90,180,365,730,1825,9125]
pairs['bucket'] = pd.cut(pairs['days'], bins)
out = pairs.groupby('bucket', observed=True)['log_ratio'].agg(
    n='size', median='median', std='std',
    extreme=lambda s: (s.abs() > 1).mean())
print(out.round(3).to_string())
print(pairs.groupby('SEGMENT')['log_ratio'].describe().round(2).to_string())

Segment rows: 1,847,060 | condo/co-op without a unit, dropped: 39,354
pairs: 395,669 (before fix: see Step 28)
                   n  median    std  extreme
bucket                                      
(-1, 0]         4571   0.295  0.698    0.160
(0, 30]         5344   0.034  0.787    0.114
(30, 90]        6924   0.229  0.812    0.104
(90, 180]      16747   0.389  0.636    0.085
(180, 365]     30209   0.390  0.685    0.108
(365, 730]     40538   0.238  0.720    0.103
(730, 1825]    95205   0.184  0.621    0.072
(1825, 9125]  196131   0.295  0.591    0.081
                count  mean   std   min   25%   50%   75%   max
SEGMENT                                                        
CONDO_COOP   188121.0  0.24  0.52 -6.48  0.05  0.22  0.43  6.49
HOUSE        188522.0  0.30  0.70 -5.75  0.04  0.31  0.59  5.76
MULTIFAMILY   19026.0  0.42  0.93 -6.52  0.05  0.37  0.75  9.07


In [10]:
pairs['ADDRESS']    = sales.loc[pairs.index, 'ADDRESS']
pairs['UNIT_FINAL'] = sales.loc[pairs.index, 'UNIT_FINAL']
pairs['extreme']    = pairs['log_ratio'].abs() > 1

# 1. Where do extremes live, by starting price? (horizons up to 2 years only)
pairs['p0_band'] = pd.cut(pairs['price0'], [0, 50_000, 100_000, 200_000, 300_000, 500_000, 1e6, 1e10])
short = pairs[(pairs['days'] > 0) & (pairs['days'] <= 730)]
print(short.groupby(['SEGMENT', 'p0_band'], observed=True)['extreme'].agg(n='size', share='mean').round(3).to_string())

# 2. What would each price floor cost and buy? (both prices must clear it)
rows = []
for f in [10_000, 50_000, 100_000, 150_000, 200_000]:
    k = pairs[(pairs['days'] > 0) & (pairs['price0'] >= f) & (pairs['price1'] >= f)]
    sh = k[k['days'] <= 730]
    rows.append({'floor': f, 'pairs_kept': len(k), 'short_n': len(sh),
                 'short_extreme': round(sh['extreme'].mean(), 3), 'short_std': round(sh['log_ratio'].std(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

# 3. Eyeball the remaining 1-30 day pairs
print(pairs[(pairs['days'] > 0) & (pairs['days'] <= 30)].sample(12, random_state=2)
      [['ADDRESS', 'UNIT_FINAL', 'SEGMENT', 'date0', 'price0', 'date1', 'price1']].to_string())

                                            n  share
SEGMENT     p0_band                                 
CONDO_COOP  (0.0, 50000.0]                481  0.599
            (50000.0, 100000.0]          1376  0.194
            (100000.0, 200000.0]         3477  0.060
            (200000.0, 300000.0]         3124  0.042
            (300000.0, 500000.0]         5006  0.046
            (500000.0, 1000000.0]        6294  0.048
            (1000000.0, 10000000000.0]   5396  0.171
HOUSE       (0.0, 50000.0]               1399  0.764
            (50000.0, 100000.0]          1131  0.720
            (100000.0, 200000.0]         7877  0.285
            (200000.0, 300000.0]        15364  0.080
            (300000.0, 500000.0]        26214  0.033
            (500000.0, 1000000.0]       15348  0.045
            (1000000.0, 10000000000.0]   2142  0.071
MULTIFAMILY (0.0, 50000.0]                201  0.761
            (50000.0, 100000.0]            88  0.557
            (100000.0, 200000.0]          210 

In [11]:
# A. Which "units" are really labels?
cc = sales[sales['SEGMENT'] == 'CONDO_COOP']
print(cc['UNIT_FINAL'].value_counts().head(25).to_string())
nodigit = cc[cc['UNIT_FINAL'].notna() & ~cc['UNIT_FINAL'].str.contains(r'\d', na=False)]
print("\nunits with no digit:", len(nodigit))
print(nodigit['UNIT_FINAL'].value_counts().head(30).to_string())

UNIT_FINAL
3A    10114
3B     9815
2A     9661
2B     9507
4A     8839
4B     8689
2      7417
1A     7321
5A     7216
3C     7158
5B     6960
1B     6896
1      6714
2C     6669
4C     6455
3D     6027
3      6009
6A     5922
6B     5779
2D     5702
5C     5685
4D     5601
5D     5104
6C     4954
3E     4734

units with no digit: 24108
UNIT_FINAL
RES      3417
A        2376
B        2276
PHA       916
PHB       829
PH        775
C         590
PHC       560
UPPER     420
D         384
PHD       382
LOWER     379
NY        333
NY,NY     315
RESI      276
CONDP     275
PHE       265
PH-A      244
PH-B      219
E         199
G         186
F         165
PHF       165
RESD      163
CONDO     148
MC        127
PH-C      123
MH        109
MA        106
PHG       106


In [12]:
# B. Relative-price rule: rebuild pairs with each sale's price vs its market median
s2 = s1[s1['SALE PRICE'] >= 10_000].copy()
s2['YEAR'] = s2['SALE DATE'].dt.year
s2['REL'] = s2['SALE PRICE'] / s2.groupby(['BOROUGH', 'SEGMENT', 'YEAR'])['SALE PRICE'].transform('median')

size = s2.groupby('MATCH_GROUP')['MATCH_GROUP'].transform('size')
s3 = s2[size >= 2].sort_values(['MATCH_GROUP', 'SALE DATE', 'SALE PRICE'])
g = s3.groupby('MATCH_GROUP')
pairs = s3[['MATCH_GROUP','SEGMENT','BOROUGH','NEIGHBORHOOD','BBL','SALE DATE','SALE PRICE','REL']].copy()
pairs['date0'] = g['SALE DATE'].shift(1); pairs['price0'] = g['SALE PRICE'].shift(1); pairs['rel0'] = g['REL'].shift(1)
pairs = pairs.dropna(subset=['date0']).rename(columns={'SALE DATE':'date1','SALE PRICE':'price1','REL':'rel1'})
pairs['days'] = (pairs['date1'] - pairs['date0']).dt.days
pairs['log_ratio'] = np.log(pairs['price1'] / pairs['price0'])
pairs['extreme'] = pairs['log_ratio'].abs() > 1
pairs['relmin'] = pairs[['rel0', 'rel1']].min(axis=1)

rows = []
for t in [0, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]:
    k = pairs[(pairs['days'] > 0) & (pairs['relmin'] >= t)]
    sh = k[k['days'] <= 730]
    rows.append({'min_rel_price': t, 'pairs_kept': len(k), 'short_n': len(sh),
                 'short_extreme': round(sh['extreme'].mean(), 3), 'short_std': round(sh['log_ratio'].std(), 3),
                 'all_extreme': round(k['extreme'].mean(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

# C. How much of the remaining extreme mass is simple fractions (1/2, 1/3, 1/4 or 2x, 3x, 4x)?
r = pairs['price1'] / pairs['price0']
frac = sum(((r / f) - 1).abs().lt(0.01) for f in [1/2, 1/3, 1/4, 2, 3, 4]) > 0
print("\nsimple-fraction price ratios:", f"extreme pairs {frac[pairs['extreme']].mean():.1%} vs other pairs {frac[~pairs['extreme']].mean():.1%}")

 min_rel_price  pairs_kept  short_n  short_extreme  short_std  all_extreme
          0.00      391098    99762          0.102      0.711        0.084
          0.10      384272    97015          0.083      0.579        0.071
          0.15      381512    96052          0.076      0.558        0.066
          0.20      378115    94855          0.069      0.543        0.061
          0.25      373252    93239          0.061      0.530        0.055
          0.30      365998    90780          0.051      0.518        0.049
          0.40      344525    83308          0.036      0.497        0.040

simple-fraction price ratios: extreme pairs 5.1% vs other pairs 1.3%


In [13]:
nd = sales.loc[(sales['SEGMENT'] == 'CONDO_COOP') & sales['UNIT_FINAL'].notna()
               & ~sales['UNIT_FINAL'].str.contains(r'\d', na=False), 'UNIT_FINAL']
print("digit-free units:", ', '.join(f"{k}:{v}" for k, v in nd.value_counts().head(40).items()))

digit-free units: RES:3417, A:2376, B:2276, PHA:916, PHB:829, PH:775, C:590, PHC:560, UPPER:420, D:384, PHD:382, LOWER:379, NY:333, NY,NY:315, RESI:276, CONDP:275, PHE:265, PH-A:244, PH-B:219, E:199, G:186, F:165, PHF:165, RESD:163, CONDO:148, MC:127, PH-C:123, MH:109, MA:106, PHG:106, MG:104, MD:104, PH-D:102, HALL:101, H:97, ME:97, MB:96, LB:95, GA:95, MF:94


In [14]:
BAD_UNITS = {'CONDO', 'COOP', 'CO-OP', 'CONDOMINIUM', 'RES', 'RESIDENTIAL', 'COMMERCIAL', 'N/A', 'NONE'}
MIN_REL, MIN_DAYS = 0.25, 90

s1 = sales[sales['SEGMENT'].notna()].copy()
bad = (s1['SEGMENT'] == 'CONDO_COOP') & (s1['UNIT_FINAL'].isna() | s1['UNIT_FINAL'].isin(BAD_UNITS))
print(f"segment rows {len(s1):,} | dropped (condo/co-op, no valid unit): {bad.sum():,}")
s1 = s1[~bad]

s2 = s1[s1['SALE PRICE'] >= 10_000].copy()
s2['YEAR'] = s2['SALE DATE'].dt.year
s2['REL'] = s2['SALE PRICE'] / s2.groupby(['BOROUGH', 'SEGMENT', 'YEAR'])['SALE PRICE'].transform('median')
n_before = len(s2)
s2 = s2[s2['REL'] >= MIN_REL]
print(f"dropped by relative-price rule: {n_before - len(s2):,}")

size = s2.groupby('MATCH_GROUP')['MATCH_GROUP'].transform('size')
s3 = s2[size >= 2].sort_values(['MATCH_GROUP', 'SALE DATE', 'SALE PRICE'])
g = s3.groupby('MATCH_GROUP')
pairs = s3[['MATCH_GROUP','SEGMENT','BOROUGH','NEIGHBORHOOD','BBL','SALE DATE','SALE PRICE']].copy()
pairs['date0'] = g['SALE DATE'].shift(1); pairs['price0'] = g['SALE PRICE'].shift(1)
pairs = pairs.dropna(subset=['date0']).rename(columns={'SALE DATE': 'date1', 'SALE PRICE': 'price1'})
pairs['days'] = (pairs['date1'] - pairs['date0']).dt.days
pairs['log_ratio'] = np.log(pairs['price1'] / pairs['price0'])
print(f"consecutive pairs: {len(pairs):,}")
pairs = pairs[pairs['days'] >= MIN_DAYS].reset_index(drop=True)

print(f"FINAL pairs: {len(pairs):,}")
print(pairs.groupby('SEGMENT')['log_ratio'].agg(n='size', median='median', std='std').round(3).to_string())
print("extreme share (|log|>1):", round((pairs['log_ratio'].abs() > 1).mean(), 3))
print(pairs.groupby(pairs['date1'].dt.year).size().to_string())

pairs.to_pickle("../data/processed/repeat_sales_pairs.pkl")

segment rows 1,847,060 | dropped (condo/co-op, no valid unit): 43,063
dropped by relative-price rule: 32,222
consecutive pairs: 376,744
FINAL pairs: 364,338
                  n  median    std
SEGMENT                           
CONDO_COOP   175660   0.227  0.414
HOUSE        172806   0.307  0.495
MULTIFAMILY   15872   0.379  0.611
extreme share (|log|>1): 0.055
date1
2003     1313
2004     4923
2005     9601
2006    12999
2007    13201
2008    11800
2009    11406
2010    11208
2011    12101
2012    14067
2013    18455
2014    19255
2015    20427
2016    21119
2017    22249
2018    21928
2019    20354
2020    16350
2021    19212
2022    25021
2023    17674
2024    19018
2025    20657


In [15]:
BAD_UNITS = {'CONDO','COOP','CO-OP','CONDOMINIUM','RES','RESI','RESD','RESIDENTIAL','CONDP',
             'COMMERCIAL','N/A','NONE','NY','NY,NY','HALL'}